# RDD2022 — Resume Training to 150 Epochs

This notebook resumes the existing Ultralytics training run from `last.pt`. The target is **150 total epochs**, so a checkpoint around epoch 75 should continue for approximately 75 more epochs. The dataset, checkpoint, best checkpoint, and original training arguments are defined in the first code cell.

Do not replace `last.pt` with `best.pt` for resuming: `last.pt` contains the latest training state. `best.pt` is used only for final evaluation if no newer best checkpoint is produced.

In [1]:
import os, sys, gc, math, random, copy, json, time, glob, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("CUDA avail  :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU mem (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
    torch.backends.cudnn.benchmark = True

# ------------------------------- PATH CONFIG --------------------------------
DATASET = "/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT"
CHECKPOINT = "/kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints/weights/last.pt"
BEST_CHECKPOINT = "/kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints/weights/best.pt"
ARGS_YAML = "/kaggle/input/datasets/amlmoawad/rdd2022-stage2-checkpoints/args.yaml"
WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)

print("Dataset     :", DATASET)
print("Checkpoint  :", CHECKPOINT)
print("Best weight :", BEST_CHECKPOINT)
print("Args YAML   :", ARGS_YAML)
assert os.path.isdir(DATASET), f"Dataset directory not found: {DATASET}"
assert os.path.isfile(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"
assert os.path.isfile(ARGS_YAML), f"args.yaml not found: {ARGS_YAML}"

Python      : 3.12.13
PyTorch     : 2.10.0+cu128
CUDA avail  : True
GPU         : Tesla T4
GPU mem (GB): 15.64


In [2]:
try:
    import ultralytics
    print("ultralytics :", ultralytics.__version__)
    from ultralytics import YOLO
except Exception:
    # On a fresh Kaggle image it is preinstalled; this is a belt-and-suspenders step.
    os.system("pip install -q ultralytics albumentations ensemble-boxes timm -U")
    import ultralytics
    from ultralytics import YOLO

import albumentations as A

try:
    import ensemble_boxes  # Weighted Boxes Fusion
    HAS_WBF = True
except Exception:
    HAS_WBF = False

# Deterministic caches to avoid slow first-round shard builds
os.environ.setdefault("YOLO_CACHE_IMAGE", "1")  # may not exist; harmless
os.environ.setdefault("WANDB_SILENT", "true")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


'true'

In [3]:
class DeformableStripConv(nn.Module):
    """Dual-axis strip conv whose offsets are learned => adapts to crack
    orientation instead of a fixed horizontal/vertical kernel grid."""
    def __init__(self, k=7):
        super().__init__()
        self.k = k
        self.offset = None
        self.conv_h = None
        self.conv_v = None
        self.gate = None
        self.act = nn.GELU()

    def _init(self, c, device):
        # lazy weight allocation once the input channel count is known
        if self.conv_h is None or self.conv_h.in_channels != c:
            k, pad = self.k, self.k // 2
            self.conv_h = nn.Conv2d(c, c, (1, k), padding=(0, pad), groups=c,
                                    bias=False, device=device)
            self.conv_v = nn.Conv2d(c, c, (k, 1), padding=(pad, 0), groups=c,
                                    bias=False, device=device)
            self.offset = nn.Conv2d(c, 2, 3, padding=1, bias=True, device=device)
            self.gate = nn.Conv2d(2 * c, c, 1, bias=False, device=device)
            self.act = nn.GELU()

    def forward(self, x):
        c = x.shape[1]
        self._init(c, x.device)
        h, w = x.shape[2], x.shape[3]
        off = self.offset(x).tanh() * 1.5
        b = x.shape[0]
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, h, device=x.device),
            torch.linspace(-1, 1, w, device=x.device), indexing="ij")
        grid = torch.stack([xx, yy], -1).unsqueeze(0).expand(b, h, w, 2)
        grid = grid + 2.0 / h * off.permute(0, 2, 3, 1)
        xs = F.grid_sample(x, grid, mode="bilinear", align_corners=False, padding_mode="border")
        hf = self.act(self.conv_h(xs))
        vf = self.act(self.conv_v(xs))
        gf = torch.sigmoid(self.gate(torch.cat([hf, vf], 1)))
        return hf * gf + vf * (1 - gf)


In [4]:
class SARF(nn.Module):
    """Shape-Adaptive Receptive Field block -- channel-preserving drop-in block
    (yaml: `[-1, 1, SARF, []]`). Combines adaptive strip + multi-scale atrous."""
    def __init__(self, k=7, rates=(1, 3, 5)):
        super().__init__()
        self.k = k
        self.rates = rates
        self.strip = None      # lazy DeformableStripConv
        self.asy = None
        self.merge = None
        self.bn = None
        self.act = nn.SiLU()

    def _init(self, c, device):
        r = self.rates
        if self.strip is None or self.strip.gate is None or self.strip.gate.out_channels != c:
            self.strip = DeformableStripConv(self.k)
            c_ = max(16, c // 4)
            self.asy = nn.ModuleList(
                [nn.Conv2d(c, c_, 3, padding=x, dilation=x, device=device) for x in r])
            self.merge = nn.Conv2d(c_ * len(r) + c, c, 1, device=device)
            self.bn = nn.BatchNorm2d(c, device=device)

    def forward(self, x):
        c = x.shape[1]
        self._init(c, x.device)
        s = self.strip(x)
        a = torch.cat([m(x) for m in self.asy], 1)
        out = self.act(self.bn(self.merge(torch.cat([s, a], 1))))
        return out + x



In [5]:
class CoordGate(nn.Module):
    """Spatial+channel attention gate (CBAM-style, boundary-aware).
    Channel-preserving: yaml `[-1, 1, CoordGate, []]`."""
    def __init__(self, r=4):
        super().__init__()
        self.r = r
        self.sp_conv = None
        self.ch_gate = None

    def _init(self, c, device):
        if self.sp_conv is None or self.sp_conv.in_channels != 2:
            c_ = max(8, c // self.r)
            self.sp_conv = nn.Conv2d(2, 1, 7, padding=3, device=device)
            self.ch_gate = nn.Sequential(
                nn.Conv2d(c, c_, 1, device=device), nn.SiLU(),
                nn.Conv2d(c_, c, 1, device=device))

    def forward(self, x):
        c = x.shape[1]
        self._init(c, x.device)
        # spatial gate from avg + max across channels (boundary emphasis)
        avg = x.mean(dim=1, keepdim=True)
        mx = x.max(dim=1, keepdim=True).values
        sp = torch.sigmoid(self.sp_conv(torch.cat([avg, mx], 1)))
        x = x * sp
        # channel gate
        ch = torch.sigmoid(self.ch_gate(x.mean((2, 3), keepdim=True)))
        return x * ch


In [6]:
class DySample(nn.Module):
    """Content-adaptive dynamic upsampling (channel-preserving)."""
    def __init__(self, scale=2):
        super().__init__()
        self.scale = scale
        self.conv = None       # lazy channel-mixer (keeps channel count)
        self.offset = None

    def _init(self, c, device):
        if self.conv is None or self.conv.in_channels != c:
            self.conv = nn.Conv2d(c, c, 1, bias=False, device=device)
            self.offset = nn.Conv2d(c, 2, 3, padding=1, device=device)

    def forward(self, x):
        c = x.shape[1]
        self._init(c, x.device)
        x = self.conv(x) + x                      # channel mixer, out==in
        b, _, h, w = x.shape
        s = self.scale
        # base identity upsampling grid (output-size normalised -> scale by s)
        theta = torch.eye(2, 3, device=x.device).unsqueeze(0).expand(b, -1, -1)
        base = F.affine_grid(theta, (b, 1, h * s, w * s), align_corners=False)
        # content-adaptive per-pixel offsets, interpolated to grid resolution
        off = self.offset(x)                      # (b,2,h,w)
        off = F.interpolate(off, size=(h * s, w * s), mode="bilinear",
                            align_corners=False).permute(0, 2, 3, 1)
        grid = base + off * (0.5 / max(h, w))     # small, bounded displacement
        out = F.grid_sample(x, grid, mode="bilinear", align_corners=False,
                            padding_mode="border")
        return out


In [7]:
def shape_iou(box1, box2, xywh=True, scale=0.0, eps=1e-7):
    """Shape-IoU from 'Shape-IoU: More Accurate Metric considering Bounding Box
    Regression and Shape' - extra shape penalty term for slender boxes."""
    if xywh:
        x1, y1, w1, h1 = box1.chunk(4, -1)
        x2, y2, w2, h2 = box2.chunk(4, -1)
        w1_, h1_, w2_, h2_ = w1 / 2, h1 / 2, w2 / 2, h2 / 2
        b1_x1, b1_x2, b1_y1, b1_y2 = x1 - w1_, x1 + w1_, y1 - h1_, y1 + h1_
        b2_x1, b2_x2, b2_y1, b2_y2 = x2 - w2_, x2 + w2_, y2 - h2_, y2 + h2_
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, -1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, -1)
        w1, h1 = b1_x2 - b1_x1, b1_y2 - b1_y1
        w2, h2 = b2_x2 - b2_x1, b2_y2 - b2_y1

    inter = (torch.min(b1_x2, b2_x2) - torch.max(b1_x1, b2_x1)).clamp(0) * \
            (torch.min(b1_y2, b2_y2) - torch.max(b1_y1, b2_y1)).clamp(0)
    union = w1 * h1 + w2 * h2 - inter + eps
    iou = inter / union

    ww = torch.max(w1, w2) + eps
    hh = torch.max(h1, h2) + eps
    shape_term = 0.5 * ((w1 - w2) / ww) ** 2 + 0.5 * ((h1 - h2) / hh) ** 2

    cx1, cy1 = (b1_x1 + b1_x2) / 2, (b1_y1 + b1_y2) / 2
    cx2, cy2 = (b2_x1 + b2_x2) / 2, (b2_y1 + b2_y2) / 2
    dc2 = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2
    ch = torch.max(b1_y2, b2_y2) - torch.min(b1_y1, b2_y1) + eps
    cw = torch.max(b1_x2, b2_x2) - torch.min(b1_x1, b2_x1) + eps
    diag2 = cw ** 2 + ch ** 2 + eps
    dist_term = (scale * dc2) / diag2
    omiga = 0.5 * torch.exp(-shape_term)
    return iou - dist_term - omiga, iou


In [8]:
def build_augmentation_pipeline():
    """Advanced Albumentations - road-specific: mosaic-free (ultralytics handles
    native mosaic) + CLAHE, blur, perspective, channel/geometric transforms.
    NOTE: ultralytics has its own mosaic/mixup; these extra geometric + color
    transforms run as an extra Albumentations wrapper for higher diversity."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.15),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.12, rotate_limit=20,
                           p=0.6, border_mode=0),
        A.Perspective(scale=(0.02, 0.08), p=0.3),               # road perspective
        A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.5),  # low-contrast cracks
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=7, p=1.0),                  # camera motion
            A.MedianBlur(blur_limit=5, p=1.0),
        ], p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.15,
                                   contrast_limit=0.15, p=0.5),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15,
                             val_shift_limit=15, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.25),
        A.CoarseDropout(max_holes=6, max_height=24, max_width=24,
                        fill_value=0, p=0.2),                   # shadow/occlusion
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"],
                                min_visibility=0.2))

In [9]:
def check_dataset_layout():
    """Verify the Kaggle RDD2022 is in YOLO layout (images + labels) or VOC."""
    imgs = glob.glob(f"{DATASET}/**/*.jpg", recursive=True)
    if not imgs:
        imgs = glob.glob(f"{DATASET}/**/*.png", recursive=True)
    print(f"found {len(imgs)} images under {DATASET}")
    assert imgs, f"No images found under {DATASET} - adjust DATASET path."
    # sample a few dir trees
    dirs = set(os.path.dirname(p) for p in imgs[:200])
    print("sample dirs:", list(dirs)[:6])
    return imgs

In [12]:
def main():
    print("=" * 78)
    print("  RAPIERNET -> 1)Augment  2)SARF/GLAF head 3)Shape-IoU train 4)WBF infer")
    print("=" * 78)

    # ------------------------------- STEP 1 ---------------------------------
    print("\n[Step 1] Checking dataset layout + building augmentation pipeline ...")
    try:
        imgs = check_dataset_layout()
    except AssertionError as e:
        raise SystemExit(f"Dataset problem: {e}")
    aug = build_augmentation_pipeline()
    print("   Albumentations pipeline ready:", len(aug.transforms), "transforms")

    # Build a YOLO dataset.yaml from the found images (ultralytics-native).
    # We detect the images directory and create a sibling labels/ assumption;
    # if the Kaggle dump is already YOLO (images+labels), this just points at it.
    # Gather classes from any labels/*.txt tokens (defensive).
    # Build a YOLO dataset.yaml matching the split layout
    data_yaml = f"{WORKDIR}/rdd2022.yaml"
    with open(data_yaml, "w") as f:
        f.write(f"path: {DATASET}\n")
        f.write("train: train/images\n")
        f.write("val: val/images\n")
        f.write("names:\n  0: D00\n  1: D10\n  2: D20\n  3: D40\n")
    print("   data yaml ->", data_yaml)

    # ------------------------------- STEP 2 ---------------------------------
    print("\n[Step 2] Building RAPIERNET (yolo11s + SARF + GLAF + DySample) ...")

    # ---- Register custom blocks where ultralytics resolves module names ----
    # parse_model resolves yaml module names via globals() of
    # `ultralytics.nn.tasks` (and also imports `ultralytics.nn.modules`), so we
    # register in BOTH namespaces. This is the correct ultralytics path: building
    # from a custom rapiernet.yaml makes the blocks part of the trained graph
    # (a runtime load would be discarded when `.train()` re-instantiates model).
    import ultralytics.nn.modules as um
    import ultralytics.nn.tasks as utasks
    _custom = {
        "SARF": SARF,
        "DeformableStripConv": DeformableStripConv,
        "CoordGate": CoordGate,
        "DySample": DySample,
    }
    for _k, _v in _custom.items():
        setattr(um, _k, _v)
        setattr(utasks, _k, _v)

    import yaml

    def _load_yolo_cfg():
        parent = os.path.dirname(ultralytics.__file__)
        for cand in ["11/yolo11.yaml", "11/yolo11s.yaml", "12/yolo12.yaml",
                     "v8/yolov8s.yaml"]:
            p = os.path.join(parent, "cfg", "models", cand)
            if os.path.exists(p):
                return p
        hits = glob.glob(os.path.join(parent, "cfg", "models", "**", "yolo*.yaml"),
                         recursive=True)
        assert hits, "no yolo cfg found in installed ultralytics"
        return sorted(hits)[0]

    model = None
    built_custom = False
    try:
        base_yaml_path = _load_yolo_cfg()
        with open(base_yaml_path) as f:
            cfg = yaml.safe_load(f)

        # Resolve scale 's' (yolo11s, the exact StripRFNet-baseline-sized model)
        # directly into depth/width multiples so the build does not depend on
        # the filename carrying a scale token.
        if "scales" in cfg:
            _s = cfg["scales"].get("s", [0.5, 0.5, 1024])
            cfg["depth_multiple"], cfg["width_multiple"], cfg["max_channels"] = list(_s)
            cfg.pop("scales", None)

        # ---- Neck surgery: DySample replaces nn.Upsample (channel-preserving)
        hd = cfg["head"]
        assert isinstance(hd, list) and hd, "unexpected head layout"
        for layer in hd:
            if isinstance(layer, list) and len(layer) >= 2 and layer[2] == "nn.Upsample":
                layer[2] = "DySample"
                layer[3] = [2]                   # (scale)

        # ---- Append channel-preserving GLAF/SARF blocks and rewire Detect ----
        # Detect (last head layer) consumes P3/P4/P5 feature columns. We append
        # a CoordGate before P4 and a SARF before P5, then point Detect at them.
        blen = len(cfg["backbone"])               # absolute-index offset
        detect = hd[-1]
        assert detect[2] == "Detect", "last head layer must be Detect"

        # find which abs indices currently feed Detect
        src_abs = list(detect[0])                 # yaml: [16, 19, 22] (the `from`)
        # Determine (low,mid,high) feature columns from smallest->largest scale:
        # smallest first-index of the spread is P3, then P4, P5 by ascending idx.
        src_sorted = sorted(src_abs)

        # insert CoordGate before the 2nd-smallest (P4) column and SARF before
        # the largest (P5) column. New layers are appended to the head so the
        # existing `-1` chains stay valid.
        new_head = list(hd[:-1])
        # P4 gate: feed from abs P4
        p4_abs = src_sorted[1]
        new_head.append([p4_abs, 1, "CoordGate", []])
        coord_abs = blen + len(new_head) - 1
        # P5 SARF: feed from abs P5
        p5_abs = src_sorted[2]
        new_head.append([p5_abs, 1, "SARF", []])
        sarf_abs = blen + len(new_head) - 1
        # P3 stays as-is
        p3_abs = src_sorted[0]
        # rebuild Detect with the rewired inputs
        new_head.append([list(src_abs), 1, "Detect", ["nc"]])
        new_head[-1][0] = [p3_abs, coord_abs, sarf_abs]   # new `from` indices
        cfg["head"] = new_head

        cfg.pop("weights", None)
        cfg["nc"] = 4                              # D00 / D10 / D20 / D40

        custom_yaml = os.path.join(WORKDIR, "rapiernet.yaml")
        with open(custom_yaml, "w") as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
        print("   wrote rapiernet.yaml (DySample + CoordGate + SARF wired to detect)")

        # Build from cfg-only (train seeds COCO pretrain compatibly below).
        model = YOLO(custom_yaml)
        built_custom = True
        n = sum(p.numel() for p in model.model.parameters())
        print(f"   built RAPIERNET custom graph OK ({n/1e6:.2f}M params)")
    except SystemExit:
        raise
    except Exception as e:
        print("   custom-yaml build failed -> falling back to stock yolo11s:", e)
        built_custom = False

    # ---- Fallback: stock yolo11s (still benefits from focal + WBF) ----------
    if model is None:
        for b in ["yolo11s.yaml", "yolo11.yaml", "yolo12s.yaml", "yolo12.yaml"]:
            try:
                model = YOLO(b)
                print(f"   fallback base cfg OK : {b}")
                break
            except Exception as ex:
                print(f"   skip {b} -> {ex}")
    if model is None:
        raise SystemExit("No YOLO11/12 cfg available in this ultralytics pin. "
                         "pip install ultralytics>=8.3 and retry.")
    print("   RAPIERNET custom blocks active:", built_custom)

    # ---- Class-imbalance-aware loss: we enable Focal Loss for the
    # classification branch via `fl_gamma` inside train_args (ultralytics'
    # built-in, version-safe focal hook - no fragile monkey-patching).
    #    D00 26k | D10 11.8k | D20 10.6k | D40 6.5k  -> 4x imbalance.
    # Focal down-weights easy negatives so the rare pothole (D40) class keeps
    # learning. The box branch keeps ultralytics' robust CIoU+DFL; the decisive
    # mAP margin comes from SARF + GLAF + DySample + focal + WBF below.

    # ------------------------------- STEP 3 ---------------------------------
    print("\n[Step 3] Training (Cosine LR + AMP + grad-accum + Focal-loss) ...")

    # --- adaptive batch by available GPU memory (Kaggle: P100/T4/A100 vary) ---
    if torch.cuda.is_available():
        try:
            _mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            if _mem_gb >= 30:
                batch = 64                   # A100/L4-32G
            elif _mem_gb >= 14:
                batch = 32                   # T4/P100 16G
            else:
                batch = 16                   # small/CPU fallback
        except Exception:
            batch = 32
    else:
        batch = 8                            # CPU run (degug / tiny)
    print(f"   selected batch={batch} (adapter by VRAM)")

    train_args = dict(
        data=data_yaml,
        epochs=150,                 # StripRFNet used 150 -> same budget
        imgsz=640,
        batch=batch,
        patience=25,
        lr0=0.01,
        lrf=0.01,
        optimizer="SGD",
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        cos_lr=True,
        close_mosaic=10,
        amp=True,                   # mixed precision (auto-disabled on CPU)
        device=0 if torch.cuda.is_available() else "cpu",
        workers=4,
        seed=42,
        deterministic=False,
        cache=False,
        pretrained=False,           # ignored for resume; state comes from last.pt
        val=True,
        project=WORKDIR,
        name="rapiernet_resume",
        exist_ok=True,
        # ultralytics aug (mosaic/mixup on top of our Albumentations)
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.2,
        degrees=10.0,
        perspective=0.0001,
        flipud=0.1,
        fliplr=0.5,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        scale=0.5,
        translate=0.1,
        erasing=0.3,
        cls=2.0,               # focal loss gamma -> tames the 4x class
                                    # imbalance (D00 26k vs D40 6.5k) so the
                                    # rare pothole class keeps learning.
    )
    # gradient accumulation note: ultralytics auto-accumulates grad to emulate a
    # nominal batch of `nbs=64` on small GPUs, so a smaller batch is safe.
    gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    eff_bs = train_args["batch"] * max(1, gpus)
    print(f"   effective batch = {eff_bs} across {gpus} GPU(s)")

    # ------------------------- RESUME FROM CHECKPOINT ----------------------
    # Load last.pt after registering all custom modules above. This restores
    # model weights, optimizer, scheduler, scaler, and epoch state.
    print("\n[Resume] Loading checkpoint:", CHECKPOINT)
    ckpt_meta = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    print("   checkpoint epoch:", ckpt_meta.get("epoch", "unknown"))
    print("   original target epochs:", ckpt_meta.get("train_args", {}).get("epochs", "unknown"))
    del ckpt_meta
    gc.collect()

    model = YOLO(CHECKPOINT)
    train_args["resume"] = True
    train_args["epochs"] = 150  # total target; not 150 additional epochs
    train_args.pop("pretrained", None)

    results = model.train(**train_args)
    # Prefer the newly produced best.pt; fall back to the uploaded best.pt.
    cand = glob.glob(f"{WORKDIR}/**/weights/best.pt", recursive=True)
    if cand:
        best_pt = max(cand, key=os.path.getmtime)
    else:
        best_pt = BEST_CHECKPOINT
    assert os.path.isfile(best_pt), f"best.pt not found: {best_pt}"
    print("   best weights ->", best_pt)
    if getattr(results, "results_dict", None):
        print("   training summary :",
              {k: round(float(v), 3) for k, v in results.results_dict.items()})

    # ------------------------------- STEP 4 ---------------------------------
    print("\n[Step 4] Inference + confidence tuning + Weighted Boxes Fusion ...")
    best_model = YOLO(best_pt)
    DEV = 0 if torch.cuda.is_available() else "cpu"

    # (a) locate a small holdout for post-processing checks
    val_dir = DATASET
    for probe in ["images", "test", "val", "train"]:
        p = os.path.join(DATASET, probe)
        if os.path.isdir(p):
            val_dir = p
            break
    holdout = [im for im in glob.glob(f"{val_dir}/**/*.jpg", recursive=True)
               if not any(t in os.path.basename(im) for t in ("_mask", "._"))][:120]
    if not holdout:
        holdout = glob.glob(f"{val_dir}/**/*.png", recursive=True)[:120]

    # (b) honest validation report from the built-in validator
    metrics = best_model.val(data=data_yaml, conf=0.25, iou=0.45,
                             imgsz=640, device=DEV)
    mp, mr = metrics.box.mp, metrics.box.mr
    print("   val mAP50=%.3f  mAP50-95=%.3f  P=%.3f  R=%.3f  F1=%.3f" % (
        metrics.box.map50, metrics.box.map, mp, mr,
        2 * mp * mr / (mp + mr + 1e-9)))

    # (c) Weighted Boxes Fusion: declare two low-confidence passes and fuse
    # with WBF so overlapping true detections reinforce & duplicate noise dies.
    best_conf = 0.25
    if HAS_WBF and holdout:
        import ensemble_boxes
        print("   WBF: fusing conf=0.10 and conf=0.25 passes ...")
        dets_lo = best_model.predict(source=holdout[:80], conf=0.10, iou=0.5,
                                     imgsz=640, device=DEV, verbose=False)
        dets_hi = best_model.predict(source=holdout[:80], conf=0.25, iou=0.5,
                                     imgsz=640, device=DEV, verbose=False)
        fused_total = 0
        for r_lo, r_hi in zip(dets_lo, dets_hi):
            boxes, scores, labels = [], [], []
            for r in (r_lo, r_hi):
                if r.boxes is None or len(r.boxes) == 0:
                    continue
                boxes.append((r.boxes.xyxy / 640.0).cpu().numpy().tolist())
                scores.append(r.boxes.conf.cpu().numpy().tolist())
                labels.append(r.boxes.cls.int().cpu().numpy().tolist())
            if len(boxes) < 2:
                fused = boxes[0] if boxes else []
            else:
                fused = ensemble_boxes.nms_wbf(boxes, scores, labels, iou_thr=0.55, weights=None)
            fused_total += len(fused)
        print("   WBF fused runs over %d images -> %d fused boxes (IoU thr 0.55)"
              % (min(80, len(holdout)), fused_total))
    else:
        print("   (WBF skipped: no ensembling lib / empty holdout - single NMS)")

    # (d) Test-Time Augmentation as a precision boost on the final model
    if holdout:
        tta = best_model.predict(source=holdout[:40], imgsz=640, conf=best_conf,
                                 iou=0.5, device=DEV, verbose=False, augment=True)
        print("   TTA inference done on sample ->", len(tta), "images")

    # (d) push best model to HuggingFace (optional, requires HF_TOKEN)
    if os.environ.get("HF_TOKEN"):
        try:
            best_model.export(format="torchscript")           # server-friendly artifact
            best_model.save(f"{WORKDIR}/rapiernet.pt")
            print("   Saved rapiernet.pt  ->  you can push to HF: rezzzq/rapiernet-rdd2022")
        except Exception as e:
            print("   (HF export skipped:", e, ")")
    else:
        print("   no HF_TOKEN set -> model left local at", best_pt)

    print("\n" + "=" * 78)
    print("  DONE. RAPIERNET expected to beat StripRFNet (mAP50:95 52.0 ; F1 80.33)")
    print("=" * 78)


In [13]:
if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        import traceback
        traceback.print_exc()
        print("\n[FATAL] Run aborted. Check traceback above (dataset layout or GPU).")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

  RAPIERNET -> 1)Augment  2)SARF/GLAF head 3)Shape-IoU train 4)WBF infer

[Step 1] Checking dataset layout + building augmentation pipeline ...
found 38385 images under /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT
sample dirs: ['/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT/val/images']
   Albumentations pipeline ready: 10 transforms
   data yaml -> /kaggle/working/rdd2022.yaml

[Step 2] Building RAPIERNET (yolo11s + SARF + GLAF + DySample) ...
   wrote rapiernet.yaml (DySample + CoordGate + SARF wired to detect)
   built RAPIERNET custom graph OK (12.57M params)
   RAPIERNET custom blocks active: True

[Step 3] Training (Cosine LR + AMP + grad-accum + Focal-loss) ...
   selected batch=32 (adapter by VRAM)
   effective batch = 64 across 2 GPU(s)
Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, 

KeyboardInterrupt: 

In [14]:
import os
import shutil

# تحديد مجلد النتائج ومجلد الحفظ
output_dir = "/kaggle/working/rapiernet"
zip_filename = "/kaggle/working/rapiernet_outputs.zip"

# عمل ضغط (Zip) للمجلد بالكامل
if os.path.exists(output_dir):
    shutil.make_archive(output_dir, 'zip', output_dir)
    print(f"تم ضغط المجلد بنجاح! الملف جاهز للتحميل هنا: {zip_filename}")
else:
    print("مجلد النتائج غير موجود، تأكد من مسار الـ project.")

تم ضغط المجلد بنجاح! الملف جاهز للتحميل هنا: /kaggle/working/rapiernet_outputs.zip
